# Credit Analyst Interview — Coding Practice

**Goal:** Build the exact skills tested in credit analyst technical interviews at banks and credit funds.

Each section mirrors a real interview task. Work through exercises in order — later exercises build on earlier ones.

> Dataset: `synthetic_portfolio_risk_data.csv` — 5,000 corporate loans

---

## Section 1 — Load and First Look
*Interviewers check if you can orient yourself in a new dataset fast.*

**Exercise 1 — Load the data**

Load `synthetic_portfolio_risk_data.csv` into a DataFrame `df`. Print:
- Shape (rows, columns)
- Default rate: what % of loans have `is_default == 1`
- Number of unique industries

> *Why this matters: In an interview you may be handed a dataset cold. The first thing a credit analyst does is understand scope and the target variable.*

**Exercise 2 — Spot data quality issues**

Check and print:
- Columns with any missing values and the count per column
- Columns that contain `inf` or `-inf` values

> *Why this matters: Real loan data is dirty. Analysts are expected to flag this before any analysis.*

---
## Section 2 — Default Rate Analysis
*Core EDA. Almost every credit analyst interview includes this.*

**Exercise 3 — Default rate by sector**

Compute default rate per `industry_sector`. Create a table with: sector, loan_count, default_rate. Sort by default_rate descending and print.

> *Why this matters: Sector concentration is the first thing a credit committee asks about.*

**Exercise 4 — Visualise sector risk**

Plot a horizontal bar chart of default rate by sector. Add a vertical dashed red line at the overall default rate. Clean labels and title.

> *Why this matters: You will present this chart in a real role.*

**Exercise 5 — Sector + leverage interaction**

Divide `debt_to_assets` into quartiles. For each (sector, D/A quartile) combination compute default rate. Pivot into a seaborn heatmap.

> *Why this matters: Interviewers test whether you control for confounders. Sector risk may just be a leverage proxy.*

---
## Section 3 — Credit Ratios
*The bread and butter of credit analysis.*

**Exercise 6 — DSCR default rate by bucket**

Cut `dscr` into 5 equal-frequency buckets using `pd.qcut`. For each bucket print the range, loan count, and default rate. Plot default rate per bucket as a bar chart.

> *Why this matters: DSCR is the most cited ratio in any credit memo. DSCR < 1.0 means the borrower cannot service its debt.*

**Exercise 7 — Interest coverage thresholds**

Replace `inf`/`-inf` in `interest_coverage` with `NaN`. Then print default rate for these four bands: ICR < 1.0, 1.0-1.5, 1.5-3.0, > 3.0.

> *Why this matters: 1.5x is a standard debt covenant threshold. Knowing where defaults spike tells you where to set the covenant.*

**Exercise 8 — Derive D/E from D/A**

Using only `debt_to_assets`, derive `debt_to_equity` from first principles:

`D/E = D/A / (1 - D/A)`

Store as `derived_de`. Verify by correlating with the existing `debt_to_equity` column.

> *Why this matters: Interviewers test accounting intuition, not just coding.*

**Exercise 9 — Liquidity trap**

Flag loans where `current_ratio` > its median AND `cash_ratio` < its median as `liquidity_trap = True`. Print: default rate for trapped vs non-trapped companies and count per group.

> *Why this matters: A company can look liquid but be illiquid if assets are stuck in inventory. This is a classic credit red flag.*

---
## Section 4 — Altman Z-Score
*The most famous quantitative credit model. Expected knowledge for any credit analyst role.*

**Exercise 10 — Compute Altman Z-Score**

Compute Z-Score for each loan and store in `altman_z`:

```
Z = 1.2*(working_capital/total_assets)
  + 1.4*(retained_earnings/total_assets)
  + 3.3*(operating_profit/total_assets)
  + 0.6*(market_cap/total_liabilities)
  + 1.0*(revenue/total_assets)
```

Classify: Safe (Z > 2.99), Grey (1.81-2.99), Distress (Z < 1.81). Print count and default rate per zone.

> *Why this matters: You may be handed a set of financials in an interview and asked to compute Z-Score from scratch.*

**Exercise 11 — Z-Score vs DSCR as standalone predictors**

Using each as a single feature, train a `LogisticRegression` to predict `is_default`. Print AUC-ROC for each. Which is the stronger predictor?

> *Why this matters: Benchmarking models is interview-level analysis. The answer (DSCR often wins) shows real understanding.*

---
## Section 5 — Portfolio Concentration
*Risk managers are obsessed with concentration. Know how to measure it.*

**Exercise 12 — HHI concentration index**

Compute the Herfindahl-Hirschman Index by sector using `total_gross_loan_amount` as exposure:

`HHI = sum((sector_exposure / total_exposure)^2)`

Print HHI and classify: < 0.15 diversified, 0.15-0.25 moderate, > 0.25 concentrated.

> *Why this matters: HHI is the standard concentration metric in banking. Portfolio managers and regulators both use it.*

**Exercise 13 — Top borrower concentration**

Sort loans by `total_gross_loan_amount` descending. Compute cumulative exposure share. Print: what % of total exposure comes from the top 10% of borrowers? Plot the cumulative curve.

> *Why this matters: "Top 10 borrowers = X% of exposure" appears in every credit committee report.*

---
## Section 6 — Expected Loss
*The core output of credit analysis. EL = PD x LGD x EAD.*

**Exercise 14 — Compute Expected Loss**

Without any ML model:
- Use the sector-level default rate as a proxy PD for each loan
- LGD = 1 - `recovery_rate` (already in the dataset)
- EAD = `total_gross_loan_amount`
- Compute `expected_loss` = PD x LGD x EAD

Print: total portfolio EL in dollars, EL as % of total exposure, and top 3 sectors by EL.

> *Why this matters: Every loan approval and regulatory filing starts here. You need to compute this in your sleep.*

**Exercise 15 — Stress test**

Apply a stress scenario: multiply all sector PDs by 1.5 (50% PD increase). Recompute total EL. Print baseline EL, stressed EL, and % increase.

> *Why this matters: CECL and Basel III both require stress testing. Showing you can run one is a strong interview signal.*

---
## Section 7 — PD Model
*Quant and model validation roles will ask you to build one.*

**Exercise 16 — Build the feature matrix**

1. Drop leakage columns: `company_id`, `industry_sector`, `is_default`, `npl_status`, `days_past_due`, `provision_for_credit_losses`
2. Drop non-numeric columns
3. Replace `inf`/`-inf` with `NaN`, then fill NaN with column median
4. Stratified 80/20 split

Print: shapes and default rate in each split.

**Exercise 17 — Logistic Regression**

Fit `LogisticRegression(max_iter=1000)` with `StandardScaler`. Print AUC-ROC, AUC-PR, and the 5 features with the largest positive coefficients.

> *Why this matters: Logistic regression is still the regulatory standard for PD models (SR 11-7). Coefficient interpretation is tested.*

**Exercise 18 — Random Forest**

Fit `RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)`. Print AUC-ROC and AUC-PR. Plot top 10 feature importances. Did it beat logistic regression?

**Exercise 19 — ROC comparison chart**

Plot ROC curves for both models on the same axes with the diagonal baseline. Label each with model name and AUC. Title: "PD Model Comparison".

> *Why this matters: You will present this chart. Clean, professional output matters.*

---
## Section 8 — Interview Scenario (Final Round)
*The kind of open-ended task given in final-round interviews.*

**Exercise 20 — Credit memo summary table**

Find the 5 riskiest loans (lowest Altman Z-Score). For each, print a clean table with:
- Company ID, Industry, Altman Z zone, DSCR, Debt-to-assets, Interest coverage, Loan amount, Expected loss

Format: 2 decimal places, dollar signs on amounts.

> *Why this matters: A credit memo is the final deliverable. This exercise combines every skill from the notebook into one output.*